In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import argparse
import json
import os
from typing import cast

import torch
import torch.nn as nn
from base import config
from models.resnet import ResNet18
from omegaconf import OmegaConf
from torch.utils.tensorboard import SummaryWriter
from utils import train_utils

from cifar import constants, load_data


In [3]:
base_config = OmegaConf.structured(config.ExperimentConfig)

In [4]:
config_path = "cifar/experiment.yaml"

In [5]:
yaml_config = OmegaConf.load(config_path)

exp_config = cast(
    config.ExperimentConfig, OmegaConf.merge(base_config, yaml_config)
)

train_config = exp_config.training
data_config = exp_config.dataset
model_config = exp_config.model

In [ ]:
train_dataloader, test_dataloader = load_data.get_dataloaders(exp_config)


/Users/peterwen/projects/learn_cv/cifar/load_data.py:39: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dict_data = pk.load(f, encoding="latin1")


In [ ]:
import torch
import torch.nn as nn

from models import config


class Block(nn.Module):
    expansion = 1

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        padding: int = 1,
        stride: int = 1,
        downsample: nn.Module | None = None,
    ):

        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.shortcut = nn.Sequential()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class Stem(nn.Module):
    def __init__(self, in_channels: int, model_config: config.ResNetStemConfig):
        super().__init__()
        self.config = model_config
        self.conv = nn.Conv2d(
            in_channels,
            out_channels=model_config.conv.out_channels,
            kernel_size=model_config.conv.kernel_size,
            stride=model_config.conv.stride,
            padding=model_config.conv.padding,
            bias=False,
        )
        self.bn = nn.BatchNorm2d(model_config.conv.out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.maxpool = nn.MaxPool2d(
            kernel_size=model_config.maxpool.kernel_size,
            stride=model_config.maxpool.stride,
            padding=model_config.maxpool.padding,
        )

    def forward(self, x):
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.maxpool(out)
        return out


class ResNet18(nn.Module):
    def __init__(
        self,
        img_size: int,
        n_classes: int,
        model_config: config.ResNet18ModelConfig,
    ):
        super().__init__()

        self.in_channels = model_config.stem.conv.out_channels
        self.stem = Stem(img_size, model_config.stem)

        layers = []
        for layer in model_config.layers:
            layers.append(
                self._make_layer(
                    Block,
                    out_channels=layer.out_channels,
                    blocks=layer.blocks,
                    stride=layer.stride,
                )
            )

        self.layers = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(model_config.fc, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None

        print(stride)
        print(self.in_channels, out_channels, block.expansion)
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    out_channels * block.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)


model = ResNet18(constants.INPUT_CHANNELS, constants.NUM_CLASSES, model_config)
model

ResNet18(
  (stem): Stem(
    (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layers): Sequential(
    (0): Sequential(
      (0): Block(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), padding=(None, None), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), padding=(None, None), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Sequential()
      )
      (1): Block(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d

In [25]:
layers = []
for layer in model_config.layers:
    layers.append(model._make_layer(Block, out_channels=layer.out_channels, blocks=layer.blocks, stride=layer.stride))

In [29]:
model.in_channels

64

In [9]:
model.stem(X).shape

torch.Size([100, 64, 16, 16])

In [16]:
model.layers[0][0].conv1

Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), padding=(None, None), bias=False)

In [18]:
model_config["layers"]

[{'out_channels': 64, 'kernel_size': 3, 'padding': 1, 'stride': 1, 'blocks': 2}]

In [19]:
from models.resnet import Block

In [22]:
model.layers[0]

Sequential(
  (0): Block(
    (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), padding=(None, None), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), padding=(None, None), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
  (1): Block(
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (shortcut): Sequential()
  )
)